# Assignment 1

![](https://media.giphy.com/media/xT9C25UNTwfZuk85WP/giphy-downsized-large.gif)

Remember the rules of ~Fight~ Code Club:
1. ALWAYS DOCUMENT
2. Cite resources that you use (paste links)
3. Include the names people who you worked with
4. Be neat and organized

## Scrape and Clean data

Based on you proposal, scrape or collect your data:

1. One variable must be either: (40 pts)
    1. scraped from the web OR;
    2. collected from an API AND you must create one new variable that is "new" to the best of your knowledge (combination of other variables representing something new).

2. You must have at least 3 variables, but you may include as many as you want into you final dataset. Likely, you will want to include more to make graphs and regressions. (30pts)

3. You must *be able* to run a regression that makes some sense with this data (the regression doesn't have to be a complete model). Briefly describe one regression you would run with your variables. (**DO NOT** run a regression, yet). (15pts)

4. You must have one combined and cleaned dataset (15 pts)

You must submit one python notebook on how you scraped/gathered data from an api, and how you combined and cleaned you data. I should be able to run your code and reproduce your final data set.  

The other variables that you choose to include do not have to be collected by API or webscraped, but you do have to combine the files and clean the dataset with python.

Thus, you must submit:
- Your finalized data set (only one) (note: you may add more variables in the future).
- Your documented python notebook
- Any associated data files needed to produce the final dataset.

You will be evaluated on:
- Completeness of the data
- Quality of the code 
- The creativity of the new variable/webscraped data you gather

Be sure to upload ALL associated files for your code to run. I will run your code from beginning to end - make it easy for me to replicate your code.

In [1]:
import pandas as pd
import numpy as np
import requests
from pprint import pprint


In [2]:
#making the file path and importing checking it works
file_path = r"C:\Users\iakov\OneDrive\Desktop\590 Bike\JC-202509-citibike-tripdata.csv\JC-202509-citibike-tripdata.csv"

df = pd.read_csv(file_path)
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,3AAD1E1BCB6B326B,electric_bike,2025-09-04 08:36:52.378,2025-09-04 08:47:21.064,Lincoln Park,JC053,Dey St,JC065,40.724605,-74.078406,40.737711,-74.066921,casual
1,EC348BE3649505F6,electric_bike,2025-09-08 18:00:44.089,2025-09-08 18:18:48.698,Exchange Pl,JC116,Dey St,JC065,40.716366,-74.034344,40.737711,-74.066921,member
2,6639ACBA00F4DC07,classic_bike,2025-09-06 08:59:06.165,2025-09-06 14:27:30.692,Union St & Bergen Ave,JC122,Dey St,JC065,40.715750,-74.078870,40.737711,-74.066921,member
3,A10A50B94E43F61E,classic_bike,2025-09-08 18:11:33.185,2025-09-08 18:17:06.671,Bergen Ave & Sip Ave,JC109,Dey St,JC065,40.731009,-74.064437,40.737711,-74.066921,member
4,56D1E808E38A2034,electric_bike,2025-09-24 17:39:37.945,2025-09-24 18:01:08.535,Exchange Pl,JC116,Dey St,JC065,40.716366,-74.034344,40.737711,-74.066921,member


In [3]:
# creating my date column and checking the member casual column
df["started_at"] = pd.to_datetime(df["started_at"])
df["date"] = df["started_at"].dt.date

df[["started_at", "date", "member_casual"]].head()


,started_at,date,member_casual
0,2025-09-04 08:36:52.378,2025-09-04,casual
1,2025-09-08 18:00:44.089,2025-09-08,member
2,2025-09-06 08:59:06.165,2025-09-06,member
3,2025-09-08 18:11:33.185,2025-09-08,member
4,2025-09-24 17:39:37.945,2025-09-24,member


In [4]:
df["member_casual"].value_counts()

member_casual
member    87773
casual    28298
Name: count, dtype: int64

In [5]:
#because there are some rides that start in 8 month that pass over midnight and their ride legnths are messed up
df = df[(df["date"] >= pd.to_datetime("2025-09-01").date()) &
        (df["date"] <= pd.to_datetime("2025-09-30").date())]

#below is cool trick learned by CHATGBT
#first creates the groups, then .size creates it in a way that its sub indexed for each date for the counts (hence total rides) for each cat member non member then 
#the unstack turns tthem into columns 
daily_rides = (df.groupby(["date", "member_casual"]).size().unstack(fill_value=0).reset_index())

#total rides and share of rides [0,1] that are from casual users
daily_rides["total_rides"] = daily_rides["casual"] + daily_rides["member"]
daily_rides["casual_share"] = daily_rides["casual"] / daily_rides["total_rides"]

daily_rides.head()

member_casual,date,casual,member,total_rides,casual_share
0,2025-09-01,1045,1908,2953,0.353877
1,2025-09-02,684,2806,3490,0.195989
2,2025-09-03,758,3159,3917,0.193515
3,2025-09-04,775,3040,3815,0.203145
4,2025-09-05,1032,3257,4289,0.240616


In [6]:
#sanity check that i didnt evaporate the data
original_total = len(df)
aggregated_total = daily_rides["total_rides"].sum()

print(original_total,aggregated_total)

116055 116055


In [7]:
# overall average ride length per day
df["started_at"] = pd.to_datetime(df["started_at"])
df["ended_at"] = pd.to_datetime(df["ended_at"])
df["ride_length_min"] = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60
daily_avg_length = df.groupby("date")["ride_length_min"].mean().reset_index()
daily_avg_length.rename(columns={"ride_length_min": "avg_ride_length_mins"}, inplace=True)

daily_rides = pd.merge(daily_rides, daily_avg_length, on="date", how="left")

# average ride length by rider type
avg_length_by_type = (df.groupby(["date", "member_casual"])["ride_length_min"].mean().unstack(fill_value=0).reset_index())

avg_length_by_type.rename(columns={"casual": "avg_casual_ride_length_mins","member": "avg_member_ride_length_mins"}, inplace=True)

daily_rides = pd.merge(daily_rides, avg_length_by_type, on="date", how="left")

daily_rides.head()

,date,casual,member,total_rides,casual_share,avg_ride_length_mins,avg_casual_ride_length_mins,avg_member_ride_length_mins
0,2025-09-01,1045,1908,2953,0.353877,15.913581,26.109869,10.329136
1,2025-09-02,684,2806,3490,0.195989,8.736759,12.817307,7.742071
2,2025-09-03,758,3159,3917,0.193515,10.228839,14.063586,9.308694
3,2025-09-04,775,3040,3815,0.203145,11.187876,12.128371,10.948112
4,2025-09-05,1032,3257,4289,0.240616,9.471251,14.461173,7.890165


In [8]:
# Open-Meteo historical daily weather for Jersey City, Sept 2025

latitude = 40.7190
longitude = -74.0500

url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": "2025-09-01",
    "end_date": "2025-09-30",
    "daily": "temperature_2m_max,temperature_2m_mean,precipitation_sum",
    "timezone": "America/New_York"
}

response = requests.get(url, params=params)
data = response.json()

data

{'latitude': 40.738136,
 'longitude': -74.04254,
 'generationtime_ms': 0.09047985076904297,
 'utc_offset_seconds': -14400,
 'timezone': 'America/New_York',
 'timezone_abbreviation': 'GMT-4',
 'elevation': 11.0,
 'daily_units': {'time': 'iso8601',
  'temperature_2m_max': '°C',
  'temperature_2m_mean': '°C',
  'precipitation_sum': 'mm'},
 'daily': {'time': ['2025-09-01',
   '2025-09-02',
   '2025-09-03',
   '2025-09-04',
   '2025-09-05',
   '2025-09-06',
   '2025-09-07',
   '2025-09-08',
   '2025-09-09',
   '2025-09-10',
   '2025-09-11',
   '2025-09-12',
   '2025-09-13',
   '2025-09-14',
   '2025-09-15',
   '2025-09-16',
   '2025-09-17',
   '2025-09-18',
   '2025-09-19',
   '2025-09-20',
   '2025-09-21',
   '2025-09-22',
   '2025-09-23',
   '2025-09-24',
   '2025-09-25',
   '2025-09-26',
   '2025-09-27',
   '2025-09-28',
   '2025-09-29',
   '2025-09-30'],
  'temperature_2m_max': [26.6,
   26.9,
   26.4,
   25.2,
   28.5,
   30.1,
   20.8,
   21.6,
   22.2,
   20.6,
   26.9,
   27.4,
   2

In [9]:
weather_df = pd.DataFrame(data["daily"])
weather_df.head()


,time,temperature_2m_max,temperature_2m_mean,precipitation_sum
0,2025-09-01,26.6,21.6,0.0
1,2025-09-02,26.9,21.0,0.0
2,2025-09-03,26.4,20.5,0.0
3,2025-09-04,25.2,20.9,4.2
4,2025-09-05,28.5,23.4,5.1


In [10]:
weather_df = weather_df.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "temperature_2m_mean": "temp_mean",
    "precipitation_sum": "precip_mm"
})

weather_df["date"] = pd.to_datetime(weather_df["date"]).dt.date


weather_df.head()

,date,temp_max,temp_mean,precip_mm
0,2025-09-01,26.6,21.6,0.0
1,2025-09-02,26.9,21.0,0.0
2,2025-09-03,26.4,20.5,0.0
3,2025-09-04,25.2,20.9,4.2
4,2025-09-05,28.5,23.4,5.1


In [11]:
daily_rides = pd.merge(daily_rides, weather_df, on="date", how="left")
daily_rides.head()

,date,casual,member,total_rides,casual_share,avg_ride_length_mins,avg_casual_ride_length_mins,avg_member_ride_length_mins,temp_max,temp_mean,precip_mm
0,2025-09-01,1045,1908,2953,0.353877,15.913581,26.109869,10.329136,26.6,21.6,0.0
1,2025-09-02,684,2806,3490,0.195989,8.736759,12.817307,7.742071,26.9,21.0,0.0
2,2025-09-03,758,3159,3917,0.193515,10.228839,14.063586,9.308694,26.4,20.5,0.0
3,2025-09-04,775,3040,3815,0.203145,11.187876,12.128371,10.948112,25.2,20.9,4.2
4,2025-09-05,1032,3257,4289,0.240616,9.471251,14.461173,7.890165,28.5,23.4,5.1


In [14]:
daily_rides["rain_any"] = (daily_rides["precip_mm"] > 0).astype(int)
daily_rides["rain_1mm"] = (daily_rides["precip_mm"] >= 1).astype(int)
daily_rides["rain_5mm"] = (daily_rides["precip_mm"] >= 5).astype(int)
daily_rides

,date,casual,member,total_rides,casual_share,avg_ride_length_mins,avg_casual_ride_length_mins,avg_member_ride_length_mins,temp_max,temp_mean,precip_mm,rain_any,rain_1mm,rain_5mm
0,2025-09-01,1045,1908,2953,0.353877,15.913581,26.109869,10.329136,26.6,21.6,0.0,0,0,0
1,2025-09-02,684,2806,3490,0.195989,8.736759,12.817307,7.742071,26.9,21.0,0.0,0,0,0
2,2025-09-03,758,3159,3917,0.193515,10.228839,14.063586,9.308694,26.4,20.5,0.0,0,0,0
3,2025-09-04,775,3040,3815,0.203145,11.187876,12.128371,10.948112,25.2,20.9,4.2,1,1,0
4,2025-09-05,1032,3257,4289,0.240616,9.471251,14.461173,7.890165,28.5,23.4,5.1,1,1,1
5,2025-09-06,1028,2241,3269,0.314469,11.009452,13.017266,10.088420,30.1,23.7,20.4,1,1,1
6,2025-09-07,666,1751,2417,0.275548,11.435315,13.988984,10.464016,20.8,18.4,17.8,1,1,1
7,2025-09-08,669,3113,3782,0.176891,10.637693,13.161837,10.095242,21.6,17.6,0.0,0,0,0
8,2025-09-09,746,3184,3930,0.189822,9.513935,16.517387,7.873051,22.2,17.6,0.0,0,0,0
9,2025-09-10,631,2817,3448,0.183005,8.057355,11.631036,7.256860,20.6,17.8,5.0,1,1,1
